# rates-curve-nlp: results walkthrough

Loads `results/*.json` produced by `scripts/run_all.sh` and shows the figures and the headline numbers. Run the pipeline first.

In [ ]:
import json, os, glob
import pandas as pd
from IPython.display import Image, display, Markdown
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
load = lambda n: json.load(open(f'results/{n}')) if os.path.exists(f'results/{n}') else None

## 1. Curve construction: bootstrap vs kernel ridge

In [ ]:
c = json.load(open(sorted(glob.glob('results/curve_*.json'))[-1]))
print('date', c['date'], '| bootstrap max reprice', c['bootstrap_max_reprice_error_bp'], 'bp |', 'KR max reprice', round(c['kernel_ridge_max_reprice_error_bp'], 4), 'bp | roughness', c['roughness_boot'], c['roughness_kr'])
pd.DataFrame(c['leave_one_out']).set_index('T').round(2)

In [ ]:
display(Image('results/figures/curve.png'))
h = load('curvehist.json')
if h: display(pd.DataFrame(h['methods']).drop(columns=['loo_yield_bp_by_year', 'hedge_test']).round(4))
display(Image('results/figures/curvehist.png'))
ht = load('hedge_test.json')
if ht: display(pd.DataFrame({(T, k): v for T, d in ht['targets'].items() for k, v in d.items() if isinstance(v, dict) and 'std_bp' in v}).T.round(3)); display(Image('results/figures/hedge.png'))

## 2. Factor risk

In [ ]:
f = load('factors.json')
pd.DataFrame({w: {'days': v['n_days'], 'explained_3': round(v['explained'][2], 3), 'sd_level': round(v['evals_bp2_per_day'][0] ** 0.5, 1), 'sd_slope': round(v['evals_bp2_per_day'][1] ** 0.5, 1), 'sd_curv': round(v['evals_bp2_per_day'][2] ** 0.5, 1)} for w, v in f.items()}).T

In [ ]:
display(Image('results/figures/factors.png'))

## 3. Closed-form quoters: frontier, decomposition, mark-outs

In [ ]:
m = load('mm_frontier.json')
df = pd.DataFrame([{**{k: c[k] for k in ['quoter', 'gamma', 'pnl_per_day', 'pnl_5min_std', 'pnl_per_var']}, **c['decomposition']} for c in m['cells']])
df.round(3)

In [ ]:
display(Image('results/figures/frontier.png'))

## 4. NLP: statement text, the text signal and the real-day replay

In [ ]:
t = load('text_signal.json')
if t: print({k: v for k, v in t.items() if k not in ('events', 'by_bank_train')}); print(t['by_bank_train'])
if os.path.exists('results/figures/text_signal.png'): display(Image('results/figures/text_signal.png'))
n = load('nlp_signal.json')
if n: display(pd.DataFrame({(k, t): v for k, r in n['feature_sets'].items() for t, v in r.items()}).T[['corr', 'corr_ci', 'sign_hit', 'placebo_corr_p95', 'placebo_p_value']]); display(Image('results/figures/nlp.png'))

In [ ]:
e = load('events.json')
if e:
    print(e['n_days'], 'days', e['n_event_days'], 'event days', e['events_by_type'])
    display(pd.DataFrame([{**{k: c[k] for k in ['quoter', 'variant', 'pnl_per_day', 'pnl_lo', 'pnl_hi', 'pnl_5min_std', 'pnl_per_var']}, **c['decomposition']} for c in e['cells']]).round(3))
    display(Image('results/figures/events.png'))

## 6. Tests

In [ ]:
print(open('results/tests.txt').read() if os.path.exists('results/tests.txt') else 'run ./build/rcmm_tests')